# OSMnx Pickup-Delivery Route PoC — Fixed Sequence + Complex OR-Tools Optimization

This notebook contains both parts:

1. **Fixed-sequence route generation** — executive provides the visit sequence, system follows it.
2. **Complex pickup-delivery optimization** — executive provides start + multiple pickup/delivery orders, system decides the best sequence while enforcing **pickup before related delivery**.

The complex section uses **7 orders** and **OR-Tools** instead of brute force, because brute force becomes impractical when the number of orders grows.

## 1. Imports and folders

In [5]:
from pathlib import Path
import webbrowser

import pandas as pd
import networkx as nx
import osmnx as ox
import folium

print("OSMnx:", ox.__version__)

OUTPUT_DIR = Path("../data/osm/processed")
MAP_DIR = Path("../data/osm/maps")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUTPUT_DIR.resolve())
print("Map folder:", MAP_DIR.resolve())

OSMnx: 2.1.1
Output folder: D:\ailabsINDIA\Experiment_codes1\technocon-fleet-optimization\data\osm\processed
Map folder: D:\ailabsINDIA\Experiment_codes1\technocon-fleet-optimization\data\osm\maps


## 2. Download larger Kolkata road network

We use `graph_from_point` with a 35 km radius instead of only `graph_from_place("Kolkata")`, because some points may fall outside the strict Kolkata administrative boundary.

In [6]:
center_point = (22.56, 88.36)   # (latitude, longitude)
distance_m = 35000              # 35 km radius

print("Downloading road network around Kolkata...")
G = ox.graph_from_point(
    center_point=center_point,
    dist=distance_m,
    network_type="drive",
    simplify=True
)

print("Road network downloaded")
print("Nodes:", len(G.nodes))
print("Edges:", len(G.edges))

Road network downloaded
Nodes: 254595
Edges: 621115


## 3. Common helper functions

In [7]:
def route_nodes_to_polyline_from_edges(G, route_nodes):
    """Convert OSMnx route node sequence into Folium polyline coordinates using edge geometry when available."""
    polyline_coords = []

    for u, v in zip(route_nodes[:-1], route_nodes[1:]):
        edge_data = G.get_edge_data(u, v)
        if not edge_data:
            continue

        best_edge = min(edge_data.values(), key=lambda edge: edge.get("length", float("inf")))

        if "geometry" in best_edge:
            coords = list(best_edge["geometry"].coords)   # (lng, lat)
            coords_lat_lng = [(lat, lng) for lng, lat in coords]
        else:
            u_data = G.nodes[u]
            v_data = G.nodes[v]
            coords_lat_lng = [(u_data["y"], u_data["x"]), (v_data["y"], v_data["x"])]

        if polyline_coords:
            coords_lat_lng = coords_lat_lng[1:]

        polyline_coords.extend(coords_lat_lng)

    return polyline_coords


def snap_point_to_graph(G, point):
    """Snap lat/lng to nearest OSM road node. OSMnx expects X=longitude, Y=latitude."""
    nearest_node = ox.distance.nearest_nodes(G, X=point["lng"], Y=point["lat"])
    node_data = G.nodes[nearest_node]

    snap_distance_m = ox.distance.great_circle(point["lat"], point["lng"], node_data["y"], node_data["x"])

    point["nearest_node"] = nearest_node
    point["snap_distance_m"] = round(snap_distance_m, 2)
    return point


def shortest_path_nodes_and_distance(G, from_node, to_node):
    path_nodes = nx.shortest_path(G, source=from_node, target=to_node, weight="length")
    distance_m = nx.shortest_path_length(G, source=from_node, target=to_node, weight="length")
    return path_nodes, distance_m


def estimate_eta_minutes(distance_km, average_speed_kmph=25):
    return round((distance_km / average_speed_kmph) * 60, 1)

# Part A — Fixed-sequence route generation

In [8]:
fixed_route_tasks = [
    {
        "task_id": "FIXED_TASK_001",
        "vehicle_id": "WB01AB1234",
        "description": "Salt Lake to Central Kolkata delivery route",
        "sequence": [
            {"name": "Start - Salt Lake Sector V", "type": "start", "lat": 22.5797, "lng": 88.4332},
            {"name": "Pickup A - City Centre Salt Lake", "type": "pickup", "lat": 22.5877, "lng": 88.4081},
            {"name": "Pickup B - Sealdah", "type": "pickup", "lat": 22.5676, "lng": 88.3684},
            {"name": "Delivery C - Park Street", "type": "delivery", "lat": 22.5535, "lng": 88.3526},
            {"name": "Delivery D - Esplanade", "type": "delivery", "lat": 22.5646, "lng": 88.3519},
        ],
    },
    {
        "task_id": "FIXED_TASK_002",
        "vehicle_id": "WB02CD5678",
        "description": "North Kolkata to Central Kolkata delivery route",
        "sequence": [
            {"name": "Start - Shyambazar", "type": "start", "lat": 22.6013, "lng": 88.3732},
            {"name": "Pickup A - College Street", "type": "pickup", "lat": 22.5764, "lng": 88.3649},
            {"name": "Pickup B - Bowbazar", "type": "pickup", "lat": 22.5695, "lng": 88.3592},
            {"name": "Delivery C - BBD Bagh", "type": "delivery", "lat": 22.5726, "lng": 88.3477},
            {"name": "Delivery D - Maidan", "type": "delivery", "lat": 22.5601, "lng": 88.3445},
        ],
    },
    {
        "task_id": "FIXED_TASK_003",
        "vehicle_id": "WB03EF9012",
        "description": "South Kolkata pickup-delivery route",
        "sequence": [
            {"name": "Start - Ballygunge", "type": "start", "lat": 22.5295, "lng": 88.3656},
            {"name": "Pickup A - Gariahat", "type": "pickup", "lat": 22.5196, "lng": 88.3657},
            {"name": "Pickup B - Jadavpur", "type": "pickup", "lat": 22.4990, "lng": 88.3718},
            {"name": "Delivery C - Tollygunge", "type": "delivery", "lat": 22.4969, "lng": 88.3453},
            {"name": "Delivery D - Kalighat", "type": "delivery", "lat": 22.5205, "lng": 88.3425},
        ],
    },
]

print("Fixed-sequence tasks:", len(fixed_route_tasks))
for task in fixed_route_tasks:
    print(task["task_id"], "-", task["description"])

Fixed-sequence tasks: 3
FIXED_TASK_001 - Salt Lake to Central Kolkata delivery route
FIXED_TASK_002 - North Kolkata to Central Kolkata delivery route
FIXED_TASK_003 - South Kolkata pickup-delivery route


In [9]:
def build_fixed_sequence_route(G, task):
    sequence = [point.copy() for point in task["sequence"]]
    for point in sequence:
        snap_point_to_graph(G, point)

    full_route_nodes = []
    segment_summaries = []
    total_distance_m = 0

    for i in range(len(sequence) - 1):
        from_point = sequence[i]
        to_point = sequence[i + 1]

        segment_nodes, segment_distance_m = shortest_path_nodes_and_distance(
            G, from_point["nearest_node"], to_point["nearest_node"]
        )
        total_distance_m += segment_distance_m

        segment_summaries.append({
            "from": from_point["name"],
            "to": to_point["name"],
            "distance_km": round(segment_distance_m / 1000, 2),
        })

        if i == 0:
            full_route_nodes.extend(segment_nodes)
        else:
            full_route_nodes.extend(segment_nodes[1:])

    route_polyline = route_nodes_to_polyline_from_edges(G, full_route_nodes)
    total_distance_km = round(total_distance_m / 1000, 2)

    return {
        "task_id": task["task_id"],
        "vehicle_id": task["vehicle_id"],
        "description": task["description"],
        "sequence": sequence,
        "route_nodes": full_route_nodes,
        "route_polyline": route_polyline,
        "total_distance_m": total_distance_m,
        "total_distance_km": total_distance_km,
        "estimated_time_min": estimate_eta_minutes(total_distance_km),
        "segment_summaries": segment_summaries,
    }

In [10]:
fixed_route_results = []
for task in fixed_route_tasks:
    print("Building fixed route for:", task["task_id"])
    fixed_route_results.append(build_fixed_sequence_route(G, task))

for result in fixed_route_results:
    print(result["task_id"], "|", result["total_distance_km"], "km | ETA", result["estimated_time_min"], "min")

Building fixed route for: FIXED_TASK_001
Building fixed route for: FIXED_TASK_002
Building fixed route for: FIXED_TASK_003
FIXED_TASK_001 | 15.31 km | ETA 36.7 min
FIXED_TASK_002 | 8.56 km | ETA 20.5 min
FIXED_TASK_003 | 11.54 km | ETA 27.7 min


In [11]:
m_fixed = folium.Map(location=[22.56, 88.36], zoom_start=12, tiles="CartoDB positron")

fixed_colors = {"FIXED_TASK_001": "blue", "FIXED_TASK_002": "green", "FIXED_TASK_003": "red"}

for result in fixed_route_results:
    task_id = result["task_id"]
    color = fixed_colors.get(task_id, "blue")

    folium.PolyLine(
        locations=result["route_polyline"],
        color=color,
        weight=5,
        opacity=0.8,
        tooltip=f"{task_id} | {result['total_distance_km']} km | ETA {result['estimated_time_min']} min",
    ).add_to(m_fixed)

    for idx, point in enumerate(result["sequence"]):
        folium.Marker(
            location=[point["lat"], point["lng"]],
            popup=f"<b>{task_id}</b><br>{idx+1}. {point['name']}<br>Type: {point['type']}<br>Snap: {point['snap_distance_m']} m",
            tooltip=f"{task_id}: {idx+1}. {point['type']}",
        ).add_to(m_fixed)

fixed_map_path = MAP_DIR / "fixed_sequence_multiple_routes.html"
m_fixed.save(fixed_map_path)
print("Saved fixed-sequence map at:", fixed_map_path.resolve())
webbrowser.open(fixed_map_path.resolve().as_uri())

Saved fixed-sequence map at: D:\ailabsINDIA\Experiment_codes1\technocon-fleet-optimization\data\osm\maps\fixed_sequence_multiple_routes.html


True

In [12]:
fixed_summary_df = pd.DataFrame([
    {
        "task_id": r["task_id"],
        "vehicle_id": r["vehicle_id"],
        "description": r["description"],
        "total_distance_km": r["total_distance_km"],
        "estimated_time_min": r["estimated_time_min"],
        "number_of_stops": len(r["sequence"]),
        "number_of_polyline_points": len(r["route_polyline"]),
    }
    for r in fixed_route_results
])

fixed_summary_path = OUTPUT_DIR / "fixed_sequence_route_summary.csv"
fixed_summary_df.to_csv(fixed_summary_path, index=False)
print("Saved:", fixed_summary_path.resolve())
fixed_summary_df

Saved: D:\ailabsINDIA\Experiment_codes1\technocon-fleet-optimization\data\osm\processed\fixed_sequence_route_summary.csv


,task_id,vehicle_id,description,total_distance_km,estimated_time_min,number_of_stops,number_of_polyline_points
0,FIXED_TASK_001,WB01AB1234,Salt Lake to Central Kolkata delivery route,15.31,36.7,5,549
1,FIXED_TASK_002,WB02CD5678,North Kolkata to Central Kolkata delivery route,8.56,20.5,5,251
2,FIXED_TASK_003,WB03EF9012,South Kolkata pickup-delivery route,11.54,27.7,5,489


# Part B — Complex pickup-delivery sequence optimization with OR-Tools

Here the executive does **not** provide the visiting sequence. The system optimizes the order.

This part uses **7 orders**, meaning 14 pickup/delivery stops plus the start point.

In [13]:
complex_task = {
    "task_id": "OPT_COMPLEX_TASK_001",
    "vehicle_id": "WB10OPT1234",
    "description": "Complex multi-order pickup-delivery optimization demo",
    "start": {"name": "Start - Salt Lake Sector V", "type": "start", "lat": 22.5797, "lng": 88.4332},
    "orders": [
        {"order_id": "ORDER_001", "pickup": {"name": "P1 - New Town Action Area I", "type": "pickup", "lat": 22.5798, "lng": 88.4613}, "delivery": {"name": "D1 - Park Street", "type": "delivery", "lat": 22.5535, "lng": 88.3526}},
        {"order_id": "ORDER_002", "pickup": {"name": "P2 - Shyambazar", "type": "pickup", "lat": 22.6013, "lng": 88.3732}, "delivery": {"name": "D2 - Behala", "type": "delivery", "lat": 22.4982, "lng": 88.3103}},
        {"order_id": "ORDER_003", "pickup": {"name": "P3 - Gariahat", "type": "pickup", "lat": 22.5196, "lng": 88.3657}, "delivery": {"name": "D3 - Howrah Station", "type": "delivery", "lat": 22.5839, "lng": 88.3425}},
        {"order_id": "ORDER_004", "pickup": {"name": "P4 - Sealdah", "type": "pickup", "lat": 22.5676, "lng": 88.3684}, "delivery": {"name": "D4 - Tollygunge", "type": "delivery", "lat": 22.4969, "lng": 88.3453}},
        {"order_id": "ORDER_005", "pickup": {"name": "P5 - Jadavpur", "type": "pickup", "lat": 22.4990, "lng": 88.3718}, "delivery": {"name": "D5 - Esplanade", "type": "delivery", "lat": 22.5646, "lng": 88.3519}},
        {"order_id": "ORDER_006", "pickup": {"name": "P6 - Dum Dum", "type": "pickup", "lat": 22.6420, "lng": 88.4312}, "delivery": {"name": "D6 - BBD Bagh", "type": "delivery", "lat": 22.5726, "lng": 88.3477}},
        {"order_id": "ORDER_007", "pickup": {"name": "P7 - Baranagar", "type": "pickup", "lat": 22.6419, "lng": 88.3773}, "delivery": {"name": "D7 - Ballygunge", "type": "delivery", "lat": 22.5295, "lng": 88.3656}},
    ],
}

print("Task:", complex_task["task_id"])
print("Orders:", len(complex_task["orders"]))
print("Total stops including start:", 1 + 2 * len(complex_task["orders"]))

Task: OPT_COMPLEX_TASK_001
Orders: 7
Total stops including start: 15


In [14]:
def create_stops_from_orders(task):
    stops = []
    start = task["start"].copy()
    start["stop_id"] = "START"
    start["order_id"] = None
    stops.append(start)

    for order in task["orders"]:
        pickup = order["pickup"].copy()
        pickup["stop_id"] = f"{order['order_id']}_PICKUP"
        pickup["order_id"] = order["order_id"]
        stops.append(pickup)

        delivery = order["delivery"].copy()
        delivery["stop_id"] = f"{order['order_id']}_DELIVERY"
        delivery["order_id"] = order["order_id"]
        stops.append(delivery)

    return stops

opt_stops = create_stops_from_orders(complex_task)
for idx, stop in enumerate(opt_stops):
    print(idx, "|", stop["stop_id"], "|", stop["name"], "|", stop["type"])

0 | START | Start - Salt Lake Sector V | start
1 | ORDER_001_PICKUP | P1 - New Town Action Area I | pickup
2 | ORDER_001_DELIVERY | D1 - Park Street | delivery
3 | ORDER_002_PICKUP | P2 - Shyambazar | pickup
4 | ORDER_002_DELIVERY | D2 - Behala | delivery
5 | ORDER_003_PICKUP | P3 - Gariahat | pickup
6 | ORDER_003_DELIVERY | D3 - Howrah Station | delivery
7 | ORDER_004_PICKUP | P4 - Sealdah | pickup
8 | ORDER_004_DELIVERY | D4 - Tollygunge | delivery
9 | ORDER_005_PICKUP | P5 - Jadavpur | pickup
10 | ORDER_005_DELIVERY | D5 - Esplanade | delivery
11 | ORDER_006_PICKUP | P6 - Dum Dum | pickup
12 | ORDER_006_DELIVERY | D6 - BBD Bagh | delivery
13 | ORDER_007_PICKUP | P7 - Baranagar | pickup
14 | ORDER_007_DELIVERY | D7 - Ballygunge | delivery


In [15]:
for stop in opt_stops:
    snap_point_to_graph(G, stop)

snap_df = pd.DataFrame([
    {
        "index": idx,
        "stop_id": stop["stop_id"],
        "name": stop["name"],
        "type": stop["type"],
        "order_id": stop["order_id"],
        "lat": stop["lat"],
        "lng": stop["lng"],
        "nearest_node": stop["nearest_node"],
        "snap_distance_m": stop["snap_distance_m"],
    }
    for idx, stop in enumerate(opt_stops)
])

snap_df.sort_values("snap_distance_m", ascending=False).head(10)

,index,stop_id,name,type,order_id,lat,lng,nearest_node,snap_distance_m
9,9,ORDER_005_PICKUP,P5 - Jadavpur,pickup,ORDER_005,22.4990,88.3718,1335919121,157.60
6,6,ORDER_003_DELIVERY,D3 - Howrah Station,delivery,ORDER_003,22.5839,88.3425,664454111,96.86
1,1,ORDER_001_PICKUP,P1 - New Town Action Area I,pickup,ORDER_001,22.5798,88.4613,2009202258,89.81
2,2,ORDER_001_DELIVERY,D1 - Park Street,delivery,ORDER_001,22.5535,88.3526,302478697,79.44
3,3,ORDER_002_PICKUP,P2 - Shyambazar,pickup,ORDER_002,22.6013,88.3732,9491482560,67.37
7,7,ORDER_004_PICKUP,P4 - Sealdah,pickup,ORDER_004,22.5676,88.3684,1196068689,67.18
0,0,START,Start - Salt Lake Sector V,start,NaN,22.5797,88.4332,6509465022,43.48
10,10,ORDER_005_DELIVERY,D5 - Esplanade,delivery,ORDER_005,22.5646,88.3519,1270249847,35.49
5,5,ORDER_003_PICKUP,P3 - Gariahat,pickup,ORDER_003,22.5196,88.3657,1591361790,33.97
12,12,ORDER_006_DELIVERY,D6 - BBD Bagh,delivery,ORDER_006,22.5726,88.3477,664446590,27.65


## Build road-distance matrix for OR-Tools

In [16]:
stop_ids = [stop["stop_id"] for stop in opt_stops]
stop_by_id = {stop["stop_id"]: stop for stop in opt_stops}
n = len(opt_stops)

distance_matrix_m = [[0 for _ in range(n)] for _ in range(n)]
path_cache = {}

for i, from_stop in enumerate(opt_stops):
    for j, to_stop in enumerate(opt_stops):
        if i == j:
            distance_matrix_m[i][j] = 0
            continue

        try:
            path_nodes, distance_m = shortest_path_nodes_and_distance(G, from_stop["nearest_node"], to_stop["nearest_node"])
            distance_matrix_m[i][j] = int(round(distance_m))
            path_cache[(i, j)] = path_nodes
        except Exception:
            distance_matrix_m[i][j] = 10**9
            path_cache[(i, j)] = None

distance_df = pd.DataFrame(distance_matrix_m, index=stop_ids, columns=stop_ids)
(distance_df / 1000).round(2)

,START,ORDER_001_PICKUP,ORDER_001_DELIVERY,ORDER_002_PICKUP,ORDER_002_DELIVERY,ORDER_003_PICKUP,ORDER_003_DELIVERY,ORDER_004_PICKUP,ORDER_004_DELIVERY,ORDER_005_PICKUP,ORDER_005_DELIVERY,ORDER_006_PICKUP,ORDER_006_DELIVERY,ORDER_007_PICKUP,ORDER_007_DELIVERY
START,0.00,3.64,10.84,8.05,19.04,11.57,10.86,8.72,15.64,13.29,10.53,9.04,10.99,12.47,10.82
ORDER_001_PICKUP,3.22,0.00,13.82,11.04,22.03,14.56,13.84,11.70,18.62,16.28,13.52,9.46,13.97,13.78,13.81
ORDER_001_DELIVERY,11.14,14.32,0.00,5.95,9.37,4.52,5.00,3.04,6.74,7.44,1.84,14.03,2.70,10.60,3.77
ORDER_002_PICKUP,8.50,11.55,6.30,0.00,15.01,9.65,4.98,4.39,12.73,12.58,5.14,8.07,5.39,4.64,8.90
ORDER_002_DELIVERY,19.53,22.71,9.37,15.01,0.00,7.64,12.21,12.22,4.99,8.22,10.17,23.09,10.42,19.29,8.67
ORDER_003_PICKUP,12.05,15.23,4.65,9.68,7.64,0.00,9.44,6.16,4.07,2.92,6.29,17.71,7.15,14.32,1.17
ORDER_003_DELIVERY,11.04,14.22,5.04,4.81,13.04,9.46,0.00,4.20,11.18,12.39,3.66,12.83,2.87,9.02,8.71
ORDER_004_PICKUP,8.98,12.16,2.96,4.10,12.15,6.37,3.82,0.00,9.53,9.29,2.29,12.17,2.74,8.74,5.62
ORDER_004_DELIVERY,16.22,19.40,7.01,12.78,4.95,4.33,11.24,9.55,0.00,4.18,8.08,20.86,8.95,17.43,5.39
ORDER_005_PICKUP,13.60,16.77,7.61,12.64,8.18,3.04,12.40,9.12,4.14,0.00,9.25,20.15,10.11,17.28,4.13


## Solve with OR-Tools pickup-delivery constraints

In [17]:
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2


def solve_pickup_delivery_with_ortools(distance_matrix_m, opt_stops, task, time_limit_seconds=20):
    num_locations = len(opt_stops)
    num_vehicles = 1
    depot_index = 0

    manager = pywrapcp.RoutingIndexManager(num_locations, num_vehicles, depot_index)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return distance_matrix_m[from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    routing.AddDimension(transit_callback_index, 0, 10**9, True, "Distance")
    distance_dimension = routing.GetDimensionOrDie("Distance")

    stop_index_by_id = {stop["stop_id"]: idx for idx, stop in enumerate(opt_stops)}

    for order in task["orders"]:
        pickup_node = stop_index_by_id[f"{order['order_id']}_PICKUP"]
        delivery_node = stop_index_by_id[f"{order['order_id']}_DELIVERY"]

        pickup_index = manager.NodeToIndex(pickup_node)
        delivery_index = manager.NodeToIndex(delivery_node)

        routing.AddPickupAndDelivery(pickup_index, delivery_index)
        routing.solver().Add(routing.VehicleVar(pickup_index) == routing.VehicleVar(delivery_index))
        routing.solver().Add(distance_dimension.CumulVar(pickup_index) <= distance_dimension.CumulVar(delivery_index))

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.seconds = time_limit_seconds

    solution = routing.SolveWithParameters(search_parameters)
    if solution is None:
        return None

    route_indices = []
    route_distance_m_solver = 0
    index = routing.Start(0)

    while not routing.IsEnd(index):
        node_index = manager.IndexToNode(index)
        route_indices.append(node_index)
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance_m_solver += routing.GetArcCostForVehicle(previous_index, index, 0)

    return {
        "route_indices": route_indices,
        "route_stop_ids": [opt_stops[i]["stop_id"] for i in route_indices],
        "route_distance_m_solver": route_distance_m_solver,
    }

ortools_solution = solve_pickup_delivery_with_ortools(distance_matrix_m, opt_stops, complex_task)

if ortools_solution is None:
    print("No solution found")
else:
    print("OR-Tools solution found")
    print("Route indices:", ortools_solution["route_indices"])
    print("Route stop IDs:", ortools_solution["route_stop_ids"])

OR-Tools solution found
Route indices: [0, 1, 11, 13, 3, 7, 14, 5, 9, 8, 4, 2, 10, 12, 6]
Route stop IDs: ['START', 'ORDER_001_PICKUP', 'ORDER_006_PICKUP', 'ORDER_007_PICKUP', 'ORDER_002_PICKUP', 'ORDER_004_PICKUP', 'ORDER_007_DELIVERY', 'ORDER_003_PICKUP', 'ORDER_005_PICKUP', 'ORDER_004_DELIVERY', 'ORDER_002_DELIVERY', 'ORDER_001_DELIVERY', 'ORDER_005_DELIVERY', 'ORDER_006_DELIVERY', 'ORDER_003_DELIVERY']


In [18]:
def check_pickup_before_delivery(route_stop_ids, task):
    position = {stop_id: idx for idx, stop_id in enumerate(route_stop_ids)}
    checks = []

    for order in task["orders"]:
        pickup_id = f"{order['order_id']}_PICKUP"
        delivery_id = f"{order['order_id']}_DELIVERY"
        pickup_pos = position.get(pickup_id)
        delivery_pos = position.get(delivery_id)
        is_valid = pickup_pos is not None and delivery_pos is not None and pickup_pos < delivery_pos

        checks.append({
            "order_id": order["order_id"],
            "pickup_stop_id": pickup_id,
            "delivery_stop_id": delivery_id,
            "pickup_position": pickup_pos,
            "delivery_position": delivery_pos,
            "pickup_before_delivery": is_valid,
        })

    return pd.DataFrame(checks)

pd_rule_check_df = check_pickup_before_delivery(ortools_solution["route_stop_ids"], complex_task)
pd_rule_check_df

,order_id,pickup_stop_id,delivery_stop_id,pickup_position,delivery_position,pickup_before_delivery
0,ORDER_001,ORDER_001_PICKUP,ORDER_001_DELIVERY,1,11,True
1,ORDER_002,ORDER_002_PICKUP,ORDER_002_DELIVERY,4,10,True
2,ORDER_003,ORDER_003_PICKUP,ORDER_003_DELIVERY,7,14,True
3,ORDER_004,ORDER_004_PICKUP,ORDER_004_DELIVERY,5,9,True
4,ORDER_005,ORDER_005_PICKUP,ORDER_005_DELIVERY,8,12,True
5,ORDER_006,ORDER_006_PICKUP,ORDER_006_DELIVERY,2,13,True
6,ORDER_007,ORDER_007_PICKUP,ORDER_007_DELIVERY,3,6,True


## Build optimized road route from OR-Tools output

In [19]:
optimized_indices = ortools_solution["route_indices"]
optimized_stops = [opt_stops[i] for i in optimized_indices]

optimized_full_route_nodes = []
optimized_segment_summaries = []
optimized_total_distance_m = 0

for a, b in zip(optimized_indices[:-1], optimized_indices[1:]):
    from_stop = opt_stops[a]
    to_stop = opt_stops[b]
    path_nodes = path_cache.get((a, b))

    if path_nodes is None:
        path_nodes, segment_distance_m = shortest_path_nodes_and_distance(G, from_stop["nearest_node"], to_stop["nearest_node"])
    else:
        segment_distance_m = distance_matrix_m[a][b]

    optimized_total_distance_m += segment_distance_m

    optimized_segment_summaries.append({
        "from_stop_id": from_stop["stop_id"],
        "to_stop_id": to_stop["stop_id"],
        "from": from_stop["name"],
        "to": to_stop["name"],
        "distance_km": round(segment_distance_m / 1000, 2),
    })

    if not optimized_full_route_nodes:
        optimized_full_route_nodes.extend(path_nodes)
    else:
        optimized_full_route_nodes.extend(path_nodes[1:])

optimized_route_polyline = route_nodes_to_polyline_from_edges(G, optimized_full_route_nodes)
optimized_total_distance_km = round(optimized_total_distance_m / 1000, 2)
optimized_eta_min = estimate_eta_minutes(optimized_total_distance_km)

print("Optimized OSM route distance:", optimized_total_distance_km, "km")
print("Estimated time:", optimized_eta_min, "min")
print("Polyline coordinate count:", len(optimized_route_polyline))

Optimized OSM route distance: 64.01 km
Estimated time: 153.6 min
Polyline coordinate count: 2311


In [20]:
optimized_order_df = pd.DataFrame([
    {
        "sequence_no": seq_no,
        "stop_id": stop["stop_id"],
        "name": stop["name"],
        "type": stop["type"],
        "order_id": stop["order_id"],
        "snap_distance_m": stop["snap_distance_m"],
    }
    for seq_no, stop in enumerate(optimized_stops, start=1)
])

optimized_order_df

,sequence_no,stop_id,name,type,order_id,snap_distance_m
0,1,START,Start - Salt Lake Sector V,start,NaN,43.48
1,2,ORDER_001_PICKUP,P1 - New Town Action Area I,pickup,ORDER_001,89.81
2,3,ORDER_006_PICKUP,P6 - Dum Dum,pickup,ORDER_006,7.26
3,4,ORDER_007_PICKUP,P7 - Baranagar,pickup,ORDER_007,5.62
4,5,ORDER_002_PICKUP,P2 - Shyambazar,pickup,ORDER_002,67.37
5,6,ORDER_004_PICKUP,P4 - Sealdah,pickup,ORDER_004,67.18
6,7,ORDER_007_DELIVERY,D7 - Ballygunge,delivery,ORDER_007,18.55
7,8,ORDER_003_PICKUP,P3 - Gariahat,pickup,ORDER_003,33.97
8,9,ORDER_005_PICKUP,P5 - Jadavpur,pickup,ORDER_005,157.60
9,10,ORDER_004_DELIVERY,D4 - Tollygunge,delivery,ORDER_004,24.21


In [21]:
m_opt = folium.Map(location=[complex_task["start"]["lat"], complex_task["start"]["lng"]], zoom_start=11, tiles="CartoDB positron")

folium.PolyLine(
    locations=optimized_route_polyline,
    color="blue",
    weight=5,
    opacity=0.85,
    tooltip=f"Optimized route | {optimized_total_distance_km} km | ETA {optimized_eta_min} min",
).add_to(m_opt)

for seq_no, stop in enumerate(optimized_stops, start=1):
    folium.Marker(
        location=[stop["lat"], stop["lng"]],
        popup=(
            f"<b>{seq_no}. {stop['name']}</b><br>"
            f"Stop ID: {stop['stop_id']}<br>"
            f"Type: {stop['type']}<br>"
            f"Order ID: {stop['order_id']}<br>"
            f"Snap distance: {stop['snap_distance_m']} m"
        ),
        tooltip=f"{seq_no}. {stop['type']} | {stop['order_id']}",
    ).add_to(m_opt)

    node_data = G.nodes[stop["nearest_node"]]
    folium.CircleMarker(
        location=[node_data["y"], node_data["x"]],
        radius=3,
        fill=True,
        tooltip=f"Snapped road node for {seq_no}",
    ).add_to(m_opt)

optimized_map_path = MAP_DIR / "complex_ortools_optimized_pickup_delivery_route.html"
m_opt.save(optimized_map_path)

print("Saved optimized map at:", optimized_map_path.resolve())
webbrowser.open(optimized_map_path.resolve().as_uri())

Saved optimized map at: D:\ailabsINDIA\Experiment_codes1\technocon-fleet-optimization\data\osm\maps\complex_ortools_optimized_pickup_delivery_route.html


True

In [22]:
optimized_segment_df = pd.DataFrame(optimized_segment_summaries)
optimized_segment_df

,from_stop_id,to_stop_id,from,to,distance_km
0,START,ORDER_001_PICKUP,Start - Salt Lake Sector V,P1 - New Town Action Area I,3.64
1,ORDER_001_PICKUP,ORDER_006_PICKUP,P1 - New Town Action Area I,P6 - Dum Dum,9.46
2,ORDER_006_PICKUP,ORDER_007_PICKUP,P6 - Dum Dum,P7 - Baranagar,7.68
3,ORDER_007_PICKUP,ORDER_002_PICKUP,P7 - Baranagar,P2 - Shyambazar,4.76
4,ORDER_002_PICKUP,ORDER_004_PICKUP,P2 - Shyambazar,P4 - Sealdah,4.39
5,ORDER_004_PICKUP,ORDER_007_DELIVERY,P4 - Sealdah,D7 - Ballygunge,5.62
6,ORDER_007_DELIVERY,ORDER_003_PICKUP,D7 - Ballygunge,P3 - Gariahat,1.28
7,ORDER_003_PICKUP,ORDER_005_PICKUP,P3 - Gariahat,P5 - Jadavpur,2.92
8,ORDER_005_PICKUP,ORDER_004_DELIVERY,P5 - Jadavpur,D4 - Tollygunge,4.14
9,ORDER_004_DELIVERY,ORDER_002_DELIVERY,D4 - Tollygunge,D2 - Behala,4.95


In [23]:
print("=" * 100)
print("Complex OR-Tools Pickup-Delivery Optimization Summary")
print("=" * 100)
print("Task:", complex_task["task_id"])
print("Vehicle:", complex_task["vehicle_id"])
print("Orders:", len(complex_task["orders"]))
print("Stops including start:", len(opt_stops))
print("Optimized distance:", optimized_total_distance_km, "km")
print("Estimated time:", optimized_eta_min, "min")
print("=" * 100)

print("\nOptimized visiting order:")
for _, row in optimized_order_df.iterrows():
    print(row["sequence_no"], "-", row["name"], "|", row["type"], "|", row["order_id"])

print("\nPickup-before-delivery rule check:")
print(pd_rule_check_df[["order_id", "pickup_position", "delivery_position", "pickup_before_delivery"]])

Complex OR-Tools Pickup-Delivery Optimization Summary
Task: OPT_COMPLEX_TASK_001
Vehicle: WB10OPT1234
Orders: 7
Stops including start: 15
Optimized distance: 64.01 km
Estimated time: 153.6 min

Optimized visiting order:
1 - Start - Salt Lake Sector V | start | nan
2 - P1 - New Town Action Area I | pickup | ORDER_001
3 - P6 - Dum Dum | pickup | ORDER_006
4 - P7 - Baranagar | pickup | ORDER_007
5 - P2 - Shyambazar | pickup | ORDER_002
6 - P4 - Sealdah | pickup | ORDER_004
7 - D7 - Ballygunge | delivery | ORDER_007
8 - P3 - Gariahat | pickup | ORDER_003
9 - P5 - Jadavpur | pickup | ORDER_005
10 - D4 - Tollygunge | delivery | ORDER_004
11 - D2 - Behala | delivery | ORDER_002
12 - D1 - Park Street | delivery | ORDER_001
13 - D5 - Esplanade | delivery | ORDER_005
14 - D6 - BBD Bagh | delivery | ORDER_006
15 - D3 - Howrah Station | delivery | ORDER_003

Pickup-before-delivery rule check:
    order_id  pickup_position  delivery_position  pickup_before_delivery
0  ORDER_001                1    

In [24]:
optimized_order_path = OUTPUT_DIR / "complex_optimized_visiting_order.csv"
optimized_segment_path = OUTPUT_DIR / "complex_optimized_segment_summary.csv"
snap_check_path = OUTPUT_DIR / "complex_optimization_snap_check.csv"

optimized_order_df.to_csv(optimized_order_path, index=False)
optimized_segment_df.to_csv(optimized_segment_path, index=False)
snap_df.to_csv(snap_check_path, index=False)

print("Saved:", optimized_order_path.resolve())
print("Saved:", optimized_segment_path.resolve())
print("Saved:", snap_check_path.resolve())

Saved: D:\ailabsINDIA\Experiment_codes1\technocon-fleet-optimization\data\osm\processed\complex_optimized_visiting_order.csv
Saved: D:\ailabsINDIA\Experiment_codes1\technocon-fleet-optimization\data\osm\processed\complex_optimized_segment_summary.csv
Saved: D:\ailabsINDIA\Experiment_codes1\technocon-fleet-optimization\data\osm\processed\complex_optimization_snap_check.csv


# Part C — PyVRP comparison demo

This section implements a **PyVRP-based version for comparison only**.

Important note:

- OR-Tools directly supports paired pickup-delivery precedence using `AddPickupAndDelivery()`.
- PyVRP is excellent for VRP/CVRP/VRPTW-style problems, but its high-level `Model` interface is not as direct for our exact paired rule:  
  **`ORDER_X_PICKUP` must happen before `ORDER_X_DELIVERY`**.
- So this section uses PyVRP to optimize the route order of all pickup/delivery stops as required clients, then checks whether the pickup-before-delivery rule is satisfied.
- If PyVRP returns a sequence that violates the rule, we apply a small **repair step** only for learning/demo purposes.

This is why OR-Tools remains our main optimizer for the current business problem.


In [25]:
# PyVRP import check

try:
    import pyvrp
    from pyvrp import Model
    from pyvrp.stop import MaxRuntime

    print("PyVRP version:", pyvrp.__version__ if hasattr(pyvrp, "__version__") else "installed")
except ImportError as exc:
    print("PyVRP is not installed in the current environment.")
    print("Install it first, then rerun this cell:")
    print("conda install -c conda-forge pyvrp -y")
    print("# or")
    print("pip install pyvrp")
    raise exc


PyVRP version: installed


## PyVRP solver function

This function uses the same inputs we already created for OR-Tools:

- `distance_matrix_m`
- `opt_stops`
- `complex_task`

But it does **not** add exact paired pickup-delivery constraints like OR-Tools does.


In [26]:
def solve_with_pyvrp_sequence_demo(distance_matrix_m, opt_stops, time_limit_seconds=20, seed=42):
    '''
    PyVRP comparison solver.

    This treats every pickup/delivery point as a required client and optimizes
    the route distance.

    It does NOT directly enforce pair-specific pickup-before-delivery constraints.
    We check and optionally repair that after solving.
    '''

    m = Model()

    # PyVRP uses x/y coordinates only for plotting/metadata here.
    # Our true travel costs come from m.add_edge(... distance=...).
    start = opt_stops[0]

    depot = m.add_depot(
        x=start["lng"],
        y=start["lat"],
        name=start["stop_id"]
    )

    clients = []
    for stop in opt_stops[1:]:
        client = m.add_client(
            x=stop["lng"],
            y=stop["lat"],
            name=stop["stop_id"]
        )
        clients.append(client)

    # One vehicle, same as the OR-Tools notebook section.
    # This creates a closed route: depot -> clients -> depot.
    m.add_vehicle_type(
        num_available=1,
        start_depot=depot,
        end_depot=depot
    )

    locations = [depot] + clients

    # Add the same road-distance matrix as explicit PyVRP edges.
    # PyVRP minimizes this distance.
    large_distance = 999_999_999

    for i, frm in enumerate(locations):
        for j, to in enumerate(locations):
            distance = int(distance_matrix_m[i][j])

            if distance >= 10**9:
                distance = large_distance

            m.add_edge(
                frm,
                to,
                distance=distance,
                duration=distance
            )

    result = m.solve(
        stop=MaxRuntime(time_limit_seconds),
        seed=seed,
        display=True
    )

    if result is None or result.best is None or len(result.best.routes()) == 0:
        return None

    # We asked for one vehicle, so normally there should be one route.
    route = result.best.routes()[0]

    # PyVRP's route.visits() usually returns client/location indices.
    raw_visits = [int(v) for v in route.visits()]

    # Version-safe conversion:
    # - If any visit is 0, assume client indices are 0-based and convert to opt_stops indices by +1.
    # - Otherwise assume PyVRP uses location indices where depot=0 and clients=1..n.
    if any(v == 0 for v in raw_visits):
        route_indices = [0] + [v + 1 for v in raw_visits]
    else:
        route_indices = [0] + raw_visits

    route_stop_ids = [opt_stops[i]["stop_id"] for i in route_indices]

    # Solver distance includes return to depot. For route comparison below,
    # we separately compute open-sequence distance using the same distance matrix.
    return {
        "pyvrp_model": m,
        "pyvrp_result": result,
        "raw_visits": raw_visits,
        "route_indices": route_indices,
        "route_stop_ids": route_stop_ids,
        "closed_route_distance_m_solver": int(route.distance()),
    }


pyvrp_solution = solve_with_pyvrp_sequence_demo(
    distance_matrix_m=distance_matrix_m,
    opt_stops=opt_stops,
    time_limit_seconds=20,
    seed=42,
)

if pyvrp_solution is None:
    print("No PyVRP solution found.")
else:
    print("PyVRP solution found.")
    print("Raw visits:", pyvrp_solution["raw_visits"])
    print("Route indices:", pyvrp_solution["route_indices"])
    print("Route stop IDs:", pyvrp_solution["route_stop_ids"])
    print("Closed-route solver distance:", round(pyvrp_solution["closed_route_distance_m_solver"] / 1000, 2), "km")


PyVRP v0.13.4

Solving an instance with:
    1 depot
    14 clients
    1 vehicle (1 vehicle type)

    Iters    Time |      Current OK    Candidate OK         Best OK
H   18954      5s |        72111  Y        72523  Y        72111  Y
    38603     10s |        72111  Y        73703  Y        72111  Y
    58787     15s |        72111  Y        72523  Y        72111  Y
    77319     22s |        72111  Y        72523  Y        72111  Y

Search terminated in 21.60s after 77319 iterations.
Best-found solution has cost 72111.

Solution results
    # routes: 1
     # trips: 1
   # clients: 14
   objective: 72111
    distance: 72111
    duration: 72111
# iterations: 77319
    run-time: 21.60 seconds
PyVRP solution found.
Raw visits: [7, 10, 2, 14, 5, 9, 8, 4, 12, 6, 3, 13, 11, 1]
Route indices: [0, 7, 10, 2, 14, 5, 9, 8, 4, 12, 6, 3, 13, 11, 1]
Route stop IDs: ['START', 'ORDER_004_PICKUP', 'ORDER_005_DELIVERY', 'ORDER_001_DELIVERY', 'ORDER_007_DELIVERY', 'ORDER_003_PICKUP', 'ORDER_005_PICKU

## Check pickup-before-delivery rule for PyVRP output

This is the key difference from OR-Tools.

OR-Tools enforces this rule during optimization.

In this PyVRP demo, we check it after solving.


In [27]:
pyvrp_rule_check_df = check_pickup_before_delivery(
    pyvrp_solution["route_stop_ids"],
    complex_task
)

pyvrp_rule_check_df


,order_id,pickup_stop_id,delivery_stop_id,pickup_position,delivery_position,pickup_before_delivery
0,ORDER_001,ORDER_001_PICKUP,ORDER_001_DELIVERY,14,3,False
1,ORDER_002,ORDER_002_PICKUP,ORDER_002_DELIVERY,11,8,False
2,ORDER_003,ORDER_003_PICKUP,ORDER_003_DELIVERY,5,10,True
3,ORDER_004,ORDER_004_PICKUP,ORDER_004_DELIVERY,1,7,True
4,ORDER_005,ORDER_005_PICKUP,ORDER_005_DELIVERY,6,2,False
5,ORDER_006,ORDER_006_PICKUP,ORDER_006_DELIVERY,13,9,False
6,ORDER_007,ORDER_007_PICKUP,ORDER_007_DELIVERY,12,4,False


In [28]:
pyvrp_rule_valid = bool(pyvrp_rule_check_df["pickup_before_delivery"].all())

print("Does raw PyVRP route satisfy pickup-before-delivery rule?", pyvrp_rule_valid)

if not pyvrp_rule_valid:
    print("Some deliveries appear before their related pickups.")
    print("For learning/demo only, we will apply a repair step next.")


Does raw PyVRP route satisfy pickup-before-delivery rule? False
Some deliveries appear before their related pickups.
For learning/demo only, we will apply a repair step next.


## Repair PyVRP sequence for pickup-before-delivery

This is not the same as native constraint optimization.

The repair step keeps the PyVRP order as much as possible but postpones deliveries until their pickup has already appeared.

This helps us visually compare PyVRP-style sequencing with the OR-Tools result, but it is not a replacement for OR-Tools' native pickup-delivery constraints.


In [29]:
def repair_pickup_before_delivery_sequence(route_stop_ids, opt_stops, task):
    '''
    Repairs a stop sequence so every pickup comes before its related delivery.

    Keeps the original route order as much as possible.
    '''
    stop_by_id = {stop["stop_id"]: stop for stop in opt_stops}

    remaining = [stop_id for stop_id in route_stop_ids if stop_id != "START"]
    repaired = ["START"]
    visited_pickup_orders = set()

    while remaining:
        progress = False

        for stop_id in list(remaining):
            stop = stop_by_id[stop_id]

            if stop["type"] == "pickup":
                repaired.append(stop_id)
                visited_pickup_orders.add(stop["order_id"])
                remaining.remove(stop_id)
                progress = True

            elif stop["type"] == "delivery":
                if stop["order_id"] in visited_pickup_orders:
                    repaired.append(stop_id)
                    remaining.remove(stop_id)
                    progress = True

        if not progress:
            raise RuntimeError(
                "Could not repair sequence. Check stop IDs and pickup/delivery order IDs."
            )

    return repaired


if pyvrp_rule_valid:
    pyvrp_repaired_route_stop_ids = pyvrp_solution["route_stop_ids"]
    print("No repair needed.")
else:
    pyvrp_repaired_route_stop_ids = repair_pickup_before_delivery_sequence(
        pyvrp_solution["route_stop_ids"],
        opt_stops,
        complex_task
    )
    print("Repair applied.")

print("PyVRP repaired/validated sequence:")
for idx, stop_id in enumerate(pyvrp_repaired_route_stop_ids, start=1):
    stop = {s["stop_id"]: s for s in opt_stops}[stop_id]
    print(idx, "-", stop_id, "|", stop["name"], "|", stop["type"], "|", stop["order_id"])


Repair applied.
PyVRP repaired/validated sequence:
1 - START | Start - Salt Lake Sector V | start | None
2 - ORDER_004_PICKUP | P4 - Sealdah | pickup | ORDER_004
3 - ORDER_003_PICKUP | P3 - Gariahat | pickup | ORDER_003
4 - ORDER_005_PICKUP | P5 - Jadavpur | pickup | ORDER_005
5 - ORDER_004_DELIVERY | D4 - Tollygunge | delivery | ORDER_004
6 - ORDER_003_DELIVERY | D3 - Howrah Station | delivery | ORDER_003
7 - ORDER_002_PICKUP | P2 - Shyambazar | pickup | ORDER_002
8 - ORDER_007_PICKUP | P7 - Baranagar | pickup | ORDER_007
9 - ORDER_006_PICKUP | P6 - Dum Dum | pickup | ORDER_006
10 - ORDER_001_PICKUP | P1 - New Town Action Area I | pickup | ORDER_001
11 - ORDER_005_DELIVERY | D5 - Esplanade | delivery | ORDER_005
12 - ORDER_001_DELIVERY | D1 - Park Street | delivery | ORDER_001
13 - ORDER_007_DELIVERY | D7 - Ballygunge | delivery | ORDER_007
14 - ORDER_002_DELIVERY | D2 - Behala | delivery | ORDER_002
15 - ORDER_006_DELIVERY | D6 - BBD Bagh | delivery | ORDER_006


In [30]:
pyvrp_repaired_rule_check_df = check_pickup_before_delivery(
    pyvrp_repaired_route_stop_ids,
    complex_task
)

pyvrp_repaired_rule_check_df


,order_id,pickup_stop_id,delivery_stop_id,pickup_position,delivery_position,pickup_before_delivery
0,ORDER_001,ORDER_001_PICKUP,ORDER_001_DELIVERY,9,11,True
1,ORDER_002,ORDER_002_PICKUP,ORDER_002_DELIVERY,6,13,True
2,ORDER_003,ORDER_003_PICKUP,ORDER_003_DELIVERY,2,5,True
3,ORDER_004,ORDER_004_PICKUP,ORDER_004_DELIVERY,1,4,True
4,ORDER_005,ORDER_005_PICKUP,ORDER_005_DELIVERY,3,10,True
5,ORDER_006,ORDER_006_PICKUP,ORDER_006_DELIVERY,8,14,True
6,ORDER_007,ORDER_007_PICKUP,ORDER_007_DELIVERY,7,12,True


## Build road route from PyVRP sequence

Now we take the PyVRP repaired/validated visiting order and generate the actual OSMnx road route, just like we did for OR-Tools.


In [31]:
def calculate_sequence_distance_from_matrix(route_indices, distance_matrix_m):
    total_distance_m = 0

    for a, b in zip(route_indices[:-1], route_indices[1:]):
        total_distance_m += distance_matrix_m[a][b]

    return total_distance_m


def build_osm_route_from_stop_ids(route_stop_ids, opt_stops, G, distance_matrix_m, path_cache):
    stop_index_by_id = {stop["stop_id"]: idx for idx, stop in enumerate(opt_stops)}
    route_indices = [stop_index_by_id[stop_id] for stop_id in route_stop_ids]

    full_route_nodes = []
    segment_summaries = []
    total_distance_m = 0

    for a, b in zip(route_indices[:-1], route_indices[1:]):
        from_stop = opt_stops[a]
        to_stop = opt_stops[b]

        path_nodes = path_cache.get((a, b))

        if path_nodes is None:
            path_nodes, segment_distance_m = shortest_path_nodes_and_distance(
                G,
                from_stop["nearest_node"],
                to_stop["nearest_node"]
            )
        else:
            segment_distance_m = distance_matrix_m[a][b]

        total_distance_m += segment_distance_m

        segment_summaries.append({
            "from_stop_id": from_stop["stop_id"],
            "to_stop_id": to_stop["stop_id"],
            "from": from_stop["name"],
            "to": to_stop["name"],
            "distance_km": round(segment_distance_m / 1000, 2),
        })

        if not full_route_nodes:
            full_route_nodes.extend(path_nodes)
        else:
            full_route_nodes.extend(path_nodes[1:])

    polyline = route_nodes_to_polyline_from_edges(G, full_route_nodes)

    return {
        "route_indices": route_indices,
        "route_stops": [opt_stops[i] for i in route_indices],
        "segment_summaries": segment_summaries,
        "total_distance_m": total_distance_m,
        "total_distance_km": round(total_distance_m / 1000, 2),
        "eta_min": estimate_eta_minutes(total_distance_m / 1000),
        "polyline": polyline,
    }


pyvrp_route_result = build_osm_route_from_stop_ids(
    route_stop_ids=pyvrp_repaired_route_stop_ids,
    opt_stops=opt_stops,
    G=G,
    distance_matrix_m=distance_matrix_m,
    path_cache=path_cache,
)

print("PyVRP repaired/validated OSM route distance:", pyvrp_route_result["total_distance_km"], "km")
print("Estimated time:", pyvrp_route_result["eta_min"], "min")
print("Polyline coordinate count:", len(pyvrp_route_result["polyline"]))


PyVRP repaired/validated OSM route distance: 97.85 km
Estimated time: 234.8 min
Polyline coordinate count: 3299


In [32]:
pyvrp_order_df = pd.DataFrame([
    {
        "sequence_no": seq_no,
        "stop_id": stop["stop_id"],
        "name": stop["name"],
        "type": stop["type"],
        "order_id": stop["order_id"],
        "snap_distance_m": stop["snap_distance_m"],
    }
    for seq_no, stop in enumerate(pyvrp_route_result["route_stops"], start=1)
])

pyvrp_order_df


,sequence_no,stop_id,name,type,order_id,snap_distance_m
0,1,START,Start - Salt Lake Sector V,start,NaN,43.48
1,2,ORDER_004_PICKUP,P4 - Sealdah,pickup,ORDER_004,67.18
2,3,ORDER_003_PICKUP,P3 - Gariahat,pickup,ORDER_003,33.97
3,4,ORDER_005_PICKUP,P5 - Jadavpur,pickup,ORDER_005,157.60
4,5,ORDER_004_DELIVERY,D4 - Tollygunge,delivery,ORDER_004,24.21
5,6,ORDER_003_DELIVERY,D3 - Howrah Station,delivery,ORDER_003,96.86
6,7,ORDER_002_PICKUP,P2 - Shyambazar,pickup,ORDER_002,67.37
7,8,ORDER_007_PICKUP,P7 - Baranagar,pickup,ORDER_007,5.62
8,9,ORDER_006_PICKUP,P6 - Dum Dum,pickup,ORDER_006,7.26
9,10,ORDER_001_PICKUP,P1 - New Town Action Area I,pickup,ORDER_001,89.81


In [33]:
pyvrp_segment_df = pd.DataFrame(pyvrp_route_result["segment_summaries"])
pyvrp_segment_df


,from_stop_id,to_stop_id,from,to,distance_km
0,START,ORDER_004_PICKUP,Start - Salt Lake Sector V,P4 - Sealdah,8.72
1,ORDER_004_PICKUP,ORDER_003_PICKUP,P4 - Sealdah,P3 - Gariahat,6.37
2,ORDER_003_PICKUP,ORDER_005_PICKUP,P3 - Gariahat,P5 - Jadavpur,2.92
3,ORDER_005_PICKUP,ORDER_004_DELIVERY,P5 - Jadavpur,D4 - Tollygunge,4.14
4,ORDER_004_DELIVERY,ORDER_003_DELIVERY,D4 - Tollygunge,D3 - Howrah Station,11.24
5,ORDER_003_DELIVERY,ORDER_002_PICKUP,D3 - Howrah Station,P2 - Shyambazar,4.81
6,ORDER_002_PICKUP,ORDER_007_PICKUP,P2 - Shyambazar,P7 - Baranagar,4.64
7,ORDER_007_PICKUP,ORDER_006_PICKUP,P7 - Baranagar,P6 - Dum Dum,7.75
8,ORDER_006_PICKUP,ORDER_001_PICKUP,P6 - Dum Dum,P1 - New Town Action Area I,9.36
9,ORDER_001_PICKUP,ORDER_005_DELIVERY,P1 - New Town Action Area I,D5 - Esplanade,13.52


## Compare OR-Tools vs PyVRP demo output

Important interpretation:

- OR-Tools result is the proper pickup-delivery constrained result.
- PyVRP result here is a route-order comparison demo.
- If PyVRP required repair, then its final distance is not a pure PyVRP-constrained optimization result.


In [34]:
comparison_rows = [
    {
        "solver": "OR-Tools",
        "native_pickup_delivery_constraint": True,
        "repair_needed": False,
        "distance_km": optimized_total_distance_km,
        "eta_min": optimized_eta_min,
        "notes": "Main optimizer for current project",
    },
    {
        "solver": "PyVRP demo",
        "native_pickup_delivery_constraint": False,
        "repair_needed": not pyvrp_rule_valid,
        "distance_km": pyvrp_route_result["total_distance_km"],
        "eta_min": pyvrp_route_result["eta_min"],
        "notes": "Sequence demo; validates/repairs pickup-before-delivery after solving",
    },
]

solver_comparison_df = pd.DataFrame(comparison_rows)
solver_comparison_df


,solver,native_pickup_delivery_constraint,repair_needed,distance_km,eta_min,notes
0,OR-Tools,True,False,64.01,153.6,Main optimizer for current project
1,PyVRP demo,False,True,97.85,234.8,Sequence demo; validates/repairs pickup-before...


## Plot PyVRP repaired/validated route


In [35]:
m_pyvrp = folium.Map(
    location=[complex_task["start"]["lat"], complex_task["start"]["lng"]],
    zoom_start=11,
    tiles="CartoDB positron"
)

folium.PolyLine(
    locations=pyvrp_route_result["polyline"],
    color="purple",
    weight=5,
    opacity=0.85,
    tooltip=(
        f"PyVRP demo route | {pyvrp_route_result['total_distance_km']} km "
        f"| ETA {pyvrp_route_result['eta_min']} min"
    ),
).add_to(m_pyvrp)

for seq_no, stop in enumerate(pyvrp_route_result["route_stops"], start=1):
    folium.Marker(
        location=[stop["lat"], stop["lng"]],
        popup=(
            f"<b>{seq_no}. {stop['name']}</b><br>"
            f"Stop ID: {stop['stop_id']}<br>"
            f"Type: {stop['type']}<br>"
            f"Order ID: {stop['order_id']}<br>"
            f"Snap distance: {stop['snap_distance_m']} m"
        ),
        tooltip=f"{seq_no}. {stop['type']} | {stop['order_id']}",
    ).add_to(m_pyvrp)

pyvrp_map_path = MAP_DIR / "complex_pyvrp_demo_pickup_delivery_route.html"
m_pyvrp.save(pyvrp_map_path)

print("Saved PyVRP demo map at:", pyvrp_map_path.resolve())
webbrowser.open(pyvrp_map_path.resolve().as_uri())


Saved PyVRP demo map at: D:\ailabsINDIA\Experiment_codes1\technocon-fleet-optimization\data\osm\maps\complex_pyvrp_demo_pickup_delivery_route.html


True

In [36]:
pyvrp_order_path = OUTPUT_DIR / "complex_pyvrp_demo_visiting_order.csv"
pyvrp_segment_path = OUTPUT_DIR / "complex_pyvrp_demo_segment_summary.csv"
solver_comparison_path = OUTPUT_DIR / "ortools_vs_pyvrp_demo_comparison.csv"

pyvrp_order_df.to_csv(pyvrp_order_path, index=False)
pyvrp_segment_df.to_csv(pyvrp_segment_path, index=False)
solver_comparison_df.to_csv(solver_comparison_path, index=False)

print("Saved:", pyvrp_order_path.resolve())
print("Saved:", pyvrp_segment_path.resolve())
print("Saved:", solver_comparison_path.resolve())


Saved: D:\ailabsINDIA\Experiment_codes1\technocon-fleet-optimization\data\osm\processed\complex_pyvrp_demo_visiting_order.csv
Saved: D:\ailabsINDIA\Experiment_codes1\technocon-fleet-optimization\data\osm\processed\complex_pyvrp_demo_segment_summary.csv
Saved: D:\ailabsINDIA\Experiment_codes1\technocon-fleet-optimization\data\osm\processed\ortools_vs_pyvrp_demo_comparison.csv


## PyVRP takeaway

PyVRP is useful and powerful for VRP-style optimisation, especially when we later want benchmark comparisons.

But for our current exact business rule:

> pickup must happen before its related delivery

OR-Tools is still cleaner because it directly supports this rule through native pickup-delivery constraints.


# Next improvements

This notebook is now complex enough for a PoC because it has 7 pickup-delivery orders and uses OR-Tools.

Next steps:

1. Add vehicle capacity.
2. Add multiple vehicles.
3. Add pickup/delivery time windows.
4. Add alert/risk penalties into the distance matrix.
5. Convert notebook logic into reusable Python modules and FastAPI APIs.